# CertCF network-complexity scaling

This notebook validates and analyzes the fixed 5 × 5 ReLU MLP grid. It reports one seed only; no across-seed error bars are implied.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULT_DIR = ROOT / 'results' / 'network_complexity'
queries = pd.read_parquet(RESULT_DIR / 'network_complexity_queries.parquet')
summary = pd.read_parquet(RESULT_DIR / 'network_complexity_summary.parquet')
assert summary['architecture_id'].nunique() == 25, 'Expected 25 complete architectures'
assert len(queries) == 25_000, 'Expected 25,000 query rows'
summary

## Complete per-architecture table

In [ ]:
table_columns = [
    'depth', 'width', 'parameter_count', 'classifier_test_accuracy',
    'build_wall_time_s', 'build_rss_peak_delta_bytes',
    'build_cuda_peak_allocated_bytes', 'validity',
    'query_time_median_s', 'query_time_p95_s', 'atlas_region_count',
]
summary[table_columns].sort_values(['depth', 'width']).style.format({
    'classifier_test_accuracy': '{:.2%}', 'validity': '{:.2%}',
    'build_wall_time_s': '{:.2f}', 'query_time_median_s': '{:.4f}',
    'query_time_p95_s': '{:.4f}',
})

## Width × depth heatmaps

In [ ]:
heatmap_metrics = {
    'build_wall_time_s': 'Atlas build time (s)',
    'build_rss_peak_delta_bytes': 'Build peak RSS increase (bytes)',
    'validity': 'Validity',
    'query_time_median_s': 'Median query time (s)',
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for ax, (metric, title) in zip(axes.flat, heatmap_metrics.items()):
    matrix = summary.pivot(index='depth', columns='width', values=metric)
    sns.heatmap(matrix, annot=True, fmt='.3g', cmap='viridis', ax=ax)
    ax.set_title(title)
plt.show()

## Build cost versus parameter count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.scatterplot(data=summary, x='parameter_count', y='build_wall_time_s', hue='depth', size='width', ax=axes[0])
sns.scatterplot(data=summary, x='parameter_count', y='build_rss_peak_delta_bytes', hue='depth', size='width', ax=axes[1])
for ax in axes:
    ax.set_xscale('log')
axes[0].set_title('Atlas build time')
axes[1].set_title('Build peak RSS increase')
plt.show()

## Marginal trends by width and depth

In [ ]:
trend_metrics = ['build_wall_time_s', 'build_rss_peak_delta_bytes', 'validity', 'query_time_median_s']
by_width = summary.groupby('width')[trend_metrics].mean(numeric_only=True)
by_depth = summary.groupby('depth')[trend_metrics].mean(numeric_only=True)
display(by_width.style.set_caption('Marginal means by width'))
display(by_depth.style.set_caption('Marginal means by depth'))
fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)
for column, metric in enumerate(trend_metrics):
    axes[0, column].plot(by_width.index, by_width[metric], marker='o')
    axes[0, column].set_title(metric)
    axes[0, column].set_xlabel('width')
    axes[1, column].plot(by_depth.index, by_depth[metric], marker='o', color='tab:orange')
    axes[1, column].set_xlabel('depth')
plt.show()

## Empirical CertCF build-time complexity

Here, CertCF *training* means atlas construction (`build_wall_time_s`), not classifier training. Candidate scaling laws are compared with leave-one-architecture-out (LOO) predictions. This is more informative than choosing a curve from in-sample $R^2$ alone.

In [ ]:
from IPython.display import Markdown
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

scaling = summary.sort_values(['depth', 'width']).reset_index(drop=True).copy()
y_build = scaling['build_wall_time_s'].to_numpy()
depth = scaling['depth'].to_numpy(dtype=float)
width = scaling['width'].to_numpy(dtype=float)
parameters = scaling['parameter_count'].to_numpy(dtype=float)
architecture = np.column_stack([depth, width])
layer_width = np.column_stack([depth, (depth - 1.0) * width])

candidate_models = {
    'Linear parameter count': (LinearRegression(), parameters[:, None], 2),
    'Power law in parameter count': (
        TransformedTargetRegressor(
            regressor=LinearRegression(), func=np.log, inverse_func=np.exp,
        ),
        np.log(parameters[:, None]),
        2,
    ),
    'Parameter count + depth': (
        LinearRegression(), np.column_stack([parameters, depth]), 3,
    ),
    'Depth + width': (LinearRegression(), architecture, 3),
    'Layer-width interaction (selected)': (
        LinearRegression(), layer_width, 3,
    ),
    'Quadratic depth/width': (
        make_pipeline(
            PolynomialFeatures(degree=2, include_bias=False),
            LinearRegression(),
        ),
        architecture,
        6,
    ),
}

loo = LeaveOneOut()
comparison_rows = []
loo_predictions = {}
for model_name, (model, features, coefficient_count) in candidate_models.items():
    prediction = cross_val_predict(model, features, y_build, cv=loo)
    fitted = model.fit(features, y_build)
    loo_predictions[model_name] = prediction
    comparison_rows.append({
        'model': model_name,
        'coefficients_including_intercept': coefficient_count,
        'LOO_MAE_s': mean_absolute_error(y_build, prediction),
        'LOO_RMSE_s': mean_squared_error(y_build, prediction) ** 0.5,
        'LOO_MAPE_pct': np.mean(np.abs((y_build - prediction) / y_build)) * 100.0,
        'LOO_predictive_R2': r2_score(y_build, prediction),
        'in_sample_R2': r2_score(y_build, fitted.predict(features)),
    })

model_comparison = (pd.DataFrame(comparison_rows)
                    .sort_values(['LOO_RMSE_s', 'coefficients_including_intercept'])
                    .reset_index(drop=True))
display(model_comparison.style.format({
    'LOO_MAE_s': '{:.2f}', 'LOO_RMSE_s': '{:.2f}',
    'LOO_MAPE_pct': '{:.2f}', 'LOO_predictive_R2': '{:.3f}',
    'in_sample_R2': '{:.3f}',
}).set_caption('Candidate build-time scaling models (lower LOO error is better)'))

selected_name = 'Layer-width interaction (selected)'
selected_model = LinearRegression().fit(layer_width, y_build)
intercept, depth_coef, layer_width_coef = (
    selected_model.intercept_, *selected_model.coef_,
)
display(Markdown(
    rf'**Selected empirical law:** $\widehat{{T}}_{{build}} = '
    rf'{intercept:.2f} + {depth_coef:.2f}D + '
    rf'{layer_width_coef:.4f}(D-1)W$ seconds.'
))

selected_loo = loo_predictions[selected_name]
prediction_frame = scaling[
    ['architecture_id', 'depth', 'width', 'parameter_count', 'build_wall_time_s']
].copy()
prediction_frame['LOO_predicted_build_time_s'] = selected_loo
prediction_frame['LOO_residual_s'] = y_build - selected_loo

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.scatterplot(
    data=prediction_frame, x='build_wall_time_s',
    y='LOO_predicted_build_time_s', hue='depth', size='width', ax=axes[0],
)
limits = [0.0, 1.05 * max(y_build.max(), selected_loo.max())]
axes[0].plot(limits, limits, '--', color='black', linewidth=1)
axes[0].set(xlim=limits, ylim=limits, title='Observed versus held-out prediction',
            xlabel='Observed build time (s)', ylabel='LOO-predicted build time (s)')
residual_grid = prediction_frame.pivot(index='depth', columns='width', values='LOO_residual_s')
sns.heatmap(residual_grid, annot=True, fmt='.1f', center=0, cmap='vlag', ax=axes[1])
axes[1].set_title('LOO residual: observed − predicted (s)')
plt.show()
prediction_frame

The quadratic model has the smallest raw LOO error, but only by a small margin and with twice as many coefficients as the selected interaction law. The selected model is preferred as the compact descriptive result: it captures the nearly width-independent cost at depth 1 and the increasing width penalty as hidden layers are added. It is an empirical interpolation over $D\in[1,5]$ and $W\in[16,256]$, not an asymptotic complexity proof. With one timing per architecture and one experimental seed, it should not be extrapolated beyond the grid or used to claim uncertainty bounds.

## Failures and resource limits

In [ ]:
failure_rows = queries.loc[~queries['success'], ['architecture_id', 'query_idx', 'error']]
failure_summary = (failure_rows.groupby(['architecture_id', 'error'], dropna=False).size()
                   .rename('count').reset_index().sort_values(['architecture_id', 'count'], ascending=[True, False]))
resource_limits = summary.loc[
    (summary['validity'] < 1.0) | ~np.isfinite(summary['build_wall_time_s']),
    ['architecture_id', 'validity', 'effective_lirpa_batch_size', 'build_wall_time_s'],
]
display(failure_summary if len(failure_summary) else pd.DataFrame({'status': ['No query failures']}))
display(resource_limits if len(resource_limits) else pd.DataFrame({'status': ['No recorded resource limits']}))